# BERTimbau na base completa (~250k) — Google Colab com TPU

Treina o `neuralmind/bert-base-portuguese-cased` na base de descrições de produtos
e mede o **teto supervisionado** (acurácia e Macro-F1) em teste estratificado intocado.
Notebook autocontido — não depende do repositório para treinar.

## Instruções (leia antes de rodar)
1. **Aloque o acelerador**: menu *Ambiente de execução → Alterar tipo de ambiente de execução*.
   - **TPU** (v2-8/v5e): funciona via `torch_xla` (já instalado no runtime TPU do Colab).
   - **GPU (A100/L4/T4)**: alternativa mais simples e frequentemente tão rápida quanto para
     um modelo do porte do BERT-base — se a TPU der problema de ambiente, troque para GPU;
     o notebook detecta sozinho e nada mais muda.
2. **Dados**: rode a célula de dados e escolha UMA das opções:
   - *Upload direto* do `dataset.csv` (colunas `nm_item`, `nm_product`); ou
   - *Google Drive* (coloque o CSV em `MeuDrive/falco/dataset.csv`).
3. Rode as células em ordem. Primeiro o **smoke** (`LIMIT=20000`, ~2-5 min) para validar
   o ambiente e medir a vazão; depois mude `LIMIT=0` (base inteira) e rode o treino de novo.
4. O relatório final (JSON) é baixado ao final — **guarde-o**: é o artefato rastreável
   do número que entra na tese (commit em `experiments/e2e3/results/`).

**Estimativas**: TPU v2-8 ou GPU A100 ≈ 10-30 min para 3 épocas na base inteira;
T4 ≈ 1-2 h. Referência a bater: PVBin ≈ 89,6% acc / Macro-F1 ≈ 0,70; teto de
medição do gabarito ≈ 99,3%.


In [ ]:
# 1) Dependências + detecção do acelerador (TPU > GPU > CPU)
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','transformers','scikit-learn'],check=True)
import torch
DEVICE_KIND='cpu'; device=torch.device('cpu'); xm=None
try:
    import torch_xla.core.xla_model as xm  # runtime TPU do Colab já traz torch_xla
    device=xm.xla_device(); DEVICE_KIND='tpu'
except Exception:
    if torch.cuda.is_available():
        device=torch.device('cuda'); DEVICE_KIND='gpu'
print('Acelerador:', DEVICE_KIND, '|', device)
if DEVICE_KIND=='tpu':
    import os; os.environ['XLA_USE_BF16']='1'  # bfloat16 nativo da TPU


In [ ]:
# 2) Dados — escolha UMA opção e comente a outra
from pathlib import Path
CSV=Path('dataset.csv')
if not CSV.exists():
    OPCAO='upload'   # 'upload' ou 'drive'
    if OPCAO=='upload':
        from google.colab import files
        up=files.upload()  # selecione o dataset.csv
        CSV=Path(next(iter(up)))
    else:
        from google.colab import drive
        drive.mount('/content/drive')
        CSV=Path('/content/drive/MyDrive/falco/dataset.csv')
print('CSV:', CSV, CSV.stat().st_size//1024, 'KB')


In [ ]:
# 3) Parâmetros — comece com o smoke; depois LIMIT=0 para a base inteira
LIMIT=20000        # 0 = base inteira (~250k)
EPOCHS=3
BATCH=128 if DEVICE_KIND!='cpu' else 16
MAX_LEN=32         # descrições têm 4-50 caracteres
LR=5e-5
SEED=42
TEST_SIZE=0.10     # teste estratificado intocado
MIN_PER_CLASS=2


In [ ]:
# 4) Carga, filtro e split estratificado
import csv, random
from collections import Counter
from sklearn.model_selection import train_test_split
rows=[(r['nm_item'],r['nm_product']) for r in csv.DictReader(open(CSV,encoding='utf-8'))]
if LIMIT: rows=rows[:LIMIT]
c=Counter(l for _,l in rows)
rows=[(t,l) for t,l in rows if c[l]>=MIN_PER_CLASS]
texts=[t for t,_ in rows]; labels=[l for _,l in rows]
tr_x,te_x,tr_y,te_y=train_test_split(texts,labels,test_size=TEST_SIZE,stratify=labels,random_state=SEED)
classes=sorted(set(labels)); lab2id={c:i for i,c in enumerate(classes)}
print(f'{len(tr_x)} treino / {len(te_x)} teste / {len(classes)} classes')


In [ ]:
# 5) Tokenização e modelo
from transformers import AutoTokenizer, AutoModelForSequenceClassification
MODEL='neuralmind/bert-base-portuguese-cased'
tok=AutoTokenizer.from_pretrained(MODEL)
model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=len(classes)).to(device)
def encode(batch_texts):
    return tok(batch_texts,truncation=True,padding='max_length',max_length=MAX_LEN,return_tensors='pt')
enc=encode(tr_x)
y=torch.tensor([lab2id[l] for l in tr_y])
ds=torch.utils.data.TensorDataset(enc['input_ids'],enc['attention_mask'],y)
g=torch.Generator().manual_seed(SEED)
dl=torch.utils.data.DataLoader(ds,batch_size=BATCH,shuffle=True,generator=g,drop_last=False)
if DEVICE_KIND=='tpu':
    import torch_xla.distributed.parallel_loader as pl
    loader=lambda: pl.MpDeviceLoader(dl,device)
else:
    loader=lambda: dl
print('lotes/época:', len(dl))


In [ ]:
# 6) Treino (laço com suporte a XLA)
import time
torch.manual_seed(SEED)
opt=torch.optim.AdamW(model.parameters(),lr=LR)
model.train(); t0=time.time()
for ep in range(EPOCHS):
    tot=0.0; n=0
    for ids,mask,yy in loader():
        if DEVICE_KIND!='tpu': ids,mask,yy=ids.to(device),mask.to(device),yy.to(device)
        opt.zero_grad()
        out=model(input_ids=ids,attention_mask=mask,labels=yy)
        out.loss.backward()
        if DEVICE_KIND=='tpu': xm.optimizer_step(opt)
        else: opt.step()
        tot+=float(out.loss.detach()); n+=1
        if n%200==0: print(f'  ep{ep+1} passo {n}/{len(dl)} loss={tot/n:.4f}',flush=True)
    print(f'época {ep+1}/{EPOCHS}: loss médio {tot/max(1,n):.4f}  [{time.time()-t0:.0f}s]',flush=True)
fit_s=time.time()-t0; print(f'treino: {fit_s:.0f}s')


In [ ]:
# 7) Avaliação no teste intocado + relatório JSON
import json, numpy as np, time
from sklearn.metrics import accuracy_score, f1_score
model.eval(); preds=[]; t0=time.time()
with torch.no_grad():
    for i in range(0,len(te_x),BATCH):
        e=encode(te_x[i:i+BATCH])
        out=model(input_ids=e['input_ids'].to(device),attention_mask=e['attention_mask'].to(device))
        preds+= [classes[j] for j in out.logits.argmax(-1).cpu().numpy()]
rep={'n_train':len(tr_x),'n_test':len(te_x),'n_classes':len(classes),
     'epochs':EPOCHS,'batch_size':BATCH,'max_length':MAX_LEN,'lr':LR,'seed':SEED,
     'accuracy':round(accuracy_score(te_y,preds),4),
     'macro_f1':round(f1_score(te_y,preds,average="macro"),4),
     'fit_seconds':round(fit_s,1),'predict_seconds':round(time.time()-t0,1),
     'device':DEVICE_KIND,'limit':LIMIT}
print(json.dumps(rep,indent=2))
fn=f'bertimbau_full_{len(tr_x)}tr_{EPOCHS}ep_s{SEED}_{DEVICE_KIND}.json'
open(fn,'w').write(json.dumps(rep,indent=2))
try:
    from google.colab import files; files.download(fn)
except Exception: pass
print('relatório salvo:',fn,'-> commit em experiments/e2e3/results/ do repo activelearning')


## Próximos passos
- Rode com `LIMIT=0` (base inteira) e guarde o JSON — é o teto supervisionado do BERTimbau.
- Para o **E2** da tese: repita variando `LIMIT ∈ {1000, 5000, 15000, 50000, 0}` ×
  `EPOCHS ∈ {1..5}` × `SEED ∈ {0..7}` e guarde todos os JSONs.
- Problemas comuns: **OOM** → reduza `BATCH`; **TPU não detectada** → confira o tipo de
  ambiente ou troque para GPU; **aviso `classifier.weight newly initialized`** → normal
  (cabeça nova no fine-tuning).
